# <div align='center'>🌍 Cidades Inteligentes: Qualidade do Ar em Bogotá (RMCAB) </div>

![Banner](https://raw.githubusercontent.com/JamaicaStoAndre/Ci-ncia-de-dados-com-Pandas/master/bogota_air_quality_banner.png)

--- 

## 🎓 Contexto: IA na Borda - Mestrado UFSC

Este material faz parte do projeto de monitoramento da **Rede de Monitoramento da Qualidade do Ar de Bogotá (RMCAB)**. Aqui, exploramos a **Biblioteca Pandas** como a ferramenta central da **Fase B (Design e Preparação)** da nossa arquitetura padronizada de Ciência de Dados.

---

## 🧪 Introdução ao Pandas: Estruturas de Dados

Antes de ler arquivos complexos, vamos entender a base: **Series** e **DataFrames**.

In [ ]:
# 1. Criando uma Series (Uma coluna única de dados)
poluicao_pm25 = pd.Series([12.5, 45.3, 30.1, 55.4]) # Cria uma série com 4 valores flutuantes
print("Exemplo de Series (Níveis de PM2.5):")         # Título do output
print(poluicao_pm25)                                # Exibe a série na tela

# 2. Criando um DataFrame (Uma tabela de dados)
dados_simulados = {                                 # Define um dicionário com os dados
    'Estacao': ['Kennedy', 'Usme', 'Suba'],       # Coluna de texto (Categorias)
    'Nivel_PM10': [25.4, 18.2, 22.1],               # Coluna de números (Metadados)
    'Atividade': [True, True, False]               # Coluna booleana (Status)
}

df_exemplo = pd.DataFrame(dados_simulados)          # Converte o dicionário em um DataFrame real
print("\nExemplo de DataFrame de Sensores:")          # Título informativo
display(df_exemplo)                                 # Exibe a tabela organizada visualmente

## 📂 1. Leitura de Arquivos (CSV / Excel)

Agora escalamos para dados reais. O Pandas brilha na ingestão automática de grandes volumes (Big Data).

In [ ]:
import pandas as pd           # Importa pandas para manipulação de tabelas (DataFrames)
import numpy as np            # Importa numpy para operações numéricas e tratamento de NaN
import folium                 # Importa folium para criar mapas interativos
from folium.plugins import HeatMap # Importa a função de mapa de calor do folium
import re                     # Importa re para limpeza de padrões em texto
import os                     # Importa os para verificar caminhos de arquivos

def parse_dms(coor):                               # Função para converter coordenadas DMS para Decimal
    if not isinstance(coor, str): return coor       # Se o valor não for texto, retorna original
    parts = re.split(r'[^\d\w]+', coor)             # Limpa caracteres especiais da string
    degrees = float(parts[0])                       # Graus
    minutes = float(parts[1])                       # Minutos
    seconds = float(parts[2]+'.'+parts[3])          # Segundos com decitilha
    direction = parts[4]                            # Norte, Sul, Leste ou Oeste
    dec_coord = degrees + minutes / 60 + seconds / 3600 # Aplica fórmula de conversão decimal
    if direction in ['S', 'W']:                     # Se for hemisfério Sul ou Oeste...
        dec_coord *= -1                             # ...valor deve ser negativo
    return dec_coord                                # Retorna coordenada final

USE_DRIVE = False                                   # Controle de armazenamento (Nuvem vs Local)
if USE_DRIVE:                                       # Se habilitado...
    from google.colab import drive                 # Habilita suporte ao Google Drive
    drive.mount('/content/drive')                  # Monta o drive para leitura no Colab
    csv_path = '/content/drive/MyDrive/1-MestradoUFSC/IA_NA_BORDA/bogota_sensors_sample.csv'
else:                                               # Caso use upload manual no Colab lateral
    opcoes = ['bogota_sensors_sample.csv', 'Bogota_sensors_sample.csv'] 
    csv_path = next((p for p in opcoes if os.path.exists(p)), 'bogota_sensors_sample.csv')

if os.path.exists(csv_path):                        # Se o arquivo for encontrado...
    df = pd.read_csv(csv_path)                      # FUNÇÃO PRINCIPAL: Leitura de arquivo CSV
    # df = pd.read_excel('arquivo.xlsx')            # (Pílula: Exemplo de como ler Excel)
    df['Lat_Dec'] = df['Latitude'].apply(parse_dms) # Manipulação: Criando coluna Latitude Decimal
    df['Lon_Dec'] = df['Longitude'].apply(parse_dms) # Manipulação: Criando coluna Longitude Decimal
    print(f"✅ Dados reais carregados de: {csv_path}")
    display(df.head())
else:
    print("❗ AVISO: Dataset real não encontrado. Use o código de simulação acima!")

## 📊 2. Operações Básicas: Filtros, Agrupamentos e Estatísticas

In [ ]:
if 'df' in locals():
    # a) ESTATÍSTICAS: Resumo completo da saúde operacional dos sensores
    print("--- Estatísticas Operacionais (PM2.5, PM10) ---")
    display(df[['PM2.5', 'PM10']].describe())       # Gera count, mean, std, min, percentis e max
    
    # b) AGRUPAMENTOS (GroupBy): Descobrir a poluição média por bairro de Bogotá
    agrupado = df.groupby('Station')[['PM2.5', 'PM10']].mean() # Agrupa por estação e tira média
    print("\n--- Média de Poluição por Estação --- ")
    display(agrupado.head())
    
    # c) FILTROS: Selecionar apenas horários onde o ar estava perigoso (PM2.5 > 35)
    critico = df[df['PM2.5'] > 35]                  # Filtro lógico instantâneo
    print(f"\n🔥 Identificamos {len(critico)} eventos críticos de poluição.")
    display(critico[['DateTime', 'Station', 'PM2.5']].head())

## 🗺️ 3. Resultado Final: Mapa de Calor (Prototipando IA na Borda)

In [ ]:
if 'df' in locals():
    m = folium.Map(location=[4.65, -74.1], zoom_start=12, tiles='cartodbpositron') # Cria mapa base
    calor = [[r['Lat_Dec'], r['Lon_Dec'], r['PM2.5']] for i, r in df.iterrows()] # Lista Latitude/Longitude/PM2.5
    HeatMap(calor, radius=18).add_to(m)                   # Plota os dados térmicos no mapa
    display(m)                                            # Exibe o mapa final de Bogotá

---

## 🎓 Desafio Prático & Resolução

**Mãos à obra:**
1. Filtre apenas a estação **'KEN'** (Kennedy) e exiba a média de PM2.5 dela.
2. Crie uma nova coluna chamada **'PM_Soma'** que seja a soma de PM2.5 + PM10.

In [ ]:
if 'df' in locals():
    # 1. Filtro local e média
    media_kennedy = df[df['Station'] == 'KEN']['PM2.5'].mean() # Filtra Kennedy e extrai média
    print(f"📊 Média PM2.5 em Kennedy: {media_kennedy:.2f}")

    # 2. Manipulação: Criando nova coluna via soma de colunas existentes
    df['PM_Soma'] = df['PM2.5'] + df['PM10']        # Operação aritmética entre vetores de dados
    print("🆕 Nova coluna 'PM_Soma' criada com sucesso!")
    display(df[['Station', 'PM2.5', 'PM10', 'PM_Soma']].head()) 